# 02 — Binary Decoder, Arousal (pos_aro vs neg_aro)

**Purpose:** Reuse the *exact* pipeline from notebook 01 to ask a different
question — can we decode **positive vs negative arousal** from single-trial beta
maps? — and then compare the result to the valence decoder.

This notebook is deliberately **short**. The whole point is that once the
pipeline is right, changing the scientific question is mostly changing *which
table you load*. If you find yourself rewriting lots of code here, stop and go
copy the working pieces from notebook 01.

---

## Notebook overview

1. Load `decoder_posaro_vs_negaro.csv` (the only real change from notebook 01).
2. Rebuild `imgs`, `y`, `groups`.
3. Reuse the same `Decoder` + `LeaveOneGroupOut` setup.
4. Fit, score, and **compare to the valence decoder**.

## What you are learning

- That a decoding *pipeline* is reusable across questions — only the **labels**
  (here, via a different input table) change.
- How to compare two decoders' accuracies meaningfully.

## Why this matters scientifically

Valence and arousal are the two classic axes of affect. Decoding each separately,
with an identical pipeline, lets you ask: *is one dimension more strongly
represented in this subject's brain patterns than the other?* Holding the
pipeline fixed is what makes that comparison fair — any accuracy difference is
about the brain, not about analysis choices.

---
## Section 1 — Load the Arousal Table

Same loading pattern as notebook 01, pointed at the arousal table.

**TODO:**
- [ ] Set `subject`, build `tables_dir`.
- [ ] Read `decoder_posaro_vs_negaro.csv` into `table`.
- [ ] Confirm `table["label"].value_counts()` is 16 / 16 across both runs.

In [28]:
from pathlib import Path
import pandas as pd
import numpy as np

subject = "sub-097"
project_dir  = Path(r"C:\ManzaRotation")
decoding_dir = project_dir / "Analysis" / "outputs" / subject / "decoding"
tables_dir   = decoding_dir / "tables"

figures_dir = decoding_dir / "figures" / "arousal_decoding"
figures_dir.mkdir(exist_ok=True)

# TODO: read decoder_posaro_vs_negaro.csv into `table`
table = pd.read_csv(tables_dir / f"{subject}_posaro_negaro_catalog.csv")
# TODO: check the class balance
table["label"].value_counts()

label
neg_aro    16
pos_aro    16
Name: count, dtype: int64

---
## Section 2 — Rebuild `imgs`, `y`, `groups`

Identical to notebook 01 — the columns have the same names, so the same three
lines work. (If you wrote a helper, this is where reuse pays off.)

**TODO:**
- [ ] Build `imgs`, `y`, `groups` from `table`.
- [ ] Assert equal lengths.

In [27]:
# TODO: imgs / y / groups from the table columns
imgs = table["path"].to_list()
y = table["label"].to_list()
groups = table["groups"].to_list()
# TODO: assert len(imgs) == len(y) == len(groups)
check = pd.DataFrame({
    "img": imgs,
    "label": y,
    "group": groups,
})

print(check.head(10))
print(check["label"].value_counts())
print(check["group"].value_counts())
print(check.groupby(["group", "label"]).size())

func_deriv_dir = project_dir / "Derivatives" / subject / "func"
mask_path = func_deriv_dir / f"{subject}_task-modulate1_space-MNI152NLin2009cAsym_res-2_desc-brain_mask.nii.gz"
print(mask_path.exists())


                                                 img    label      group
0  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
1  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
2  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
3  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
4  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
5  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
6  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
7  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  neg_aro  modulate1
8  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_aro  modulate1
9  C:\ManzaRotation\Analysis\outputs\sub-097\beta...  pos_aro  modulate1
label
neg_aro    16
pos_aro    16
Name: count, dtype: int64
group
modulate1    16
modulate2    16
Name: count, dtype: int64
group      label  
modulate1  neg_aro    8
           pos_aro    8
modula

---
## Section 3 — Reuse the Same Decoder Setup

Nothing about masking, feature selection, the SVM, or cross-validation changes
when the question changes — so reuse the same configuration. Keeping
`screening_percentile`, `estimator`, `mask`, and `cv` **identical** to notebook
01 is what makes the comparison in Section 5 fair.

**TODO:**
- [ ] Point `mask_path` at the same brain mask as notebook 01.
- [ ] Build `cv = LeaveOneGroupOut()`.
- [ ] Build a `Decoder` with the **same** settings you used in notebook 01.

In [30]:
from nilearn import plotting
from pathlib import Path
import numpy as np
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut
# Make an empty list where each decoder result will be stored


results = []
cv = LeaveOneGroupOut()
# Loop over different feature-screening percentages
for screening_percentile in [1, 2, 5, 10, 25, 50]:

    # Build the decoder for this screening percentile
    decoder = Decoder(
        estimator="svc",
        mask=mask_path,
        standardize="zscore_sample",
        screening_percentile=screening_percentile,
        scoring="accuracy",
        cv=cv,
    )

    # Fit the decoder using images, labels, and run/group labels
    decoder.fit(imgs, y, groups=groups)

    # Loop over the cross-validation scores saved inside the fitted decoder
    for label, scores in decoder.cv_scores_.items():
        # TODO: calculate the mean accuracy for this label
        mean_score = np.mean(scores)
        
        # TODO: make a dictionary containing:
        # subject
        # screening_percentile
        # label
        # mean accuracy
        # fold scores
        one_result = {
            "subject": subject,
            "screening_percentile": screening_percentile,
            "label": label,
            "mean_score": mean_score,
            "fold_scores": scores
        }
        results.append(one_result)
        

    # TODO: loop over the decoder coefficient images
    for label, images in decoder.coef_img_.items():
        plotting.plot_stat_map(
            images,
            title=f"SVM weight map for {label}",
            display_mode="ortho",
            cut_coords=(0, 0, 0),
            cmap="RdBu_r",
            colorbar=True,
            threshold=0,
            output_file=figures_dir / f"{subject}_{screening_percentile}_svm_weight_map_{label}.png"
        )
results_dir = decoding_dir / "scores"
results_dir.mkdir(exist_ok=True)
results_csv = pd.DataFrame(results)
results_csv.to_csv(results_dir / f"{subject}_arousal_scores.csv")

        # TODO: plot and save the coefficient image using plotting.plot_stat_map

C:\Users\jwhit\AppData\Local\Temp\ipykernel_13636\228983879.py:25: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)
C:\Users\jwhit\AppData\Local\Temp\ipykernel_13636\228983879.py:25: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)
C:\Users\jwhit\AppData\Local\Temp\ipykernel_13636\228983879.py:25: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  decoder.fit(imgs, y, groups=groups)
C:\Users\jwhit\AppData\Local\Temp\ipykernel_13636\228983879.py:25: UserWarning: [NiftiMasker.fit] Generation of a mask has been requested (imgs != None) while a mask was given at masker creation. Given mask will be used.
  